Tác giả: <23120257> – <Bùi Trung Hiếu>

 [Giai đoạn 1] – Entry Discovery & Full-Text Crawling
------------------------------------------------------
Tác giả: <23120257> – <Bùi Trung Hiếu>

Mục tiêu:

    • Tự động phát hiện và tải các bài báo arXiv theo ID được gán.

    • Lấy metadata, version history, và toàn bộ source LaTeX (.tex, .bib).

    • Hỗ trợ đa luồng song song để tăng tốc độ crawl.

    • Tự động tạo cấu trúc thư mục đúng chuẩn nộp lab (tex/, metadata.json, ...).

Đầu ra:
    📂 <BASE_DIR>/<arxiv_id>/

        ├── tex/
        │   ├── <arxiv_id>v1/*.tex, *.bib
        │   ├── <arxiv_id>v2/*.tex, *.bib
        ├── metadata.json
    → Phục vụ trực tiếp cho bước “references.json” ở giai đoạn 2.

Thông tin kỹ thuật:
    - Ngôn ngữ: Python 3.12

    - Môi trường chạy: Google Colab (CPU)

    - Thư viện chính: arxiv, requests, bs4, concurrent.futures

    - Cấu trúc output tuân thủ quy định Data Submission trong đề bài.



In [ ]:

# Cấu hình Google Drive
from google.colab import drive
drive.mount('/content/mydrive')

# Cài thư viện cần thiết
!pip install -q arxiv beautifulsoup4 lxml requests feedparser

# ======================== PHẦN CHÍNH ========================
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# ---- Thư viện chuẩn ----
import os, tarfile, json, time, requests, feedparser, re, gzip, shutil, csv, psutil
import arxiv
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Cấu hình ----
BASE_DIR = Path("/content/mydrive/MyDrive/23120257")
BASE_DIR.mkdir(parents=True, exist_ok=True)
START_MONTH = "2024-11"
STATS_FILE = BASE_DIR / "statistics_program1.csv"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; ArxivDownloader/1.0; +https://arxiv.org)"
})

# Nếu file thống kê chưa tồn tại → tạo header
if not STATS_FILE.exists():
    with open(STATS_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "arxiv_id", "success_rate", "size_before_figures",
            "size_after_figures", "runtime_sec",
            "ram_usage_mb", "disk_usage_mb"
        ])

# ---------------------- HÀM HỖ TRỢ ----------------------
"""Tạo thư mục nếu chưa tồn tại."""
def safe_mkdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)



def fetch_versions_html(arxiv_id: str):
    """Lấy danh sách các phiên bản v1, v2, ... từ HTML arXiv."""
    url = f"https://arxiv.org/abs/{arxiv_id}"
    try:
        res = session.get(url, timeout=10)
        if res.status_code != 200:
            return ["v1"]
        soup = BeautifulSoup(res.text, "html.parser")
        history = soup.find("div", {"class": "submission-history"})
        if not history:
            return ["v1"]
         # Regex tìm các ký hiệu [v1], [v2], ...
        versions = re.findall(r"\[(v\d+)\]", history.text)
        return versions or ["v1"]
    except Exception:
        return ["v1"]


"""Lấy các ngày revised (v2 trở đi)."""
def fetch_revised_dates(arxiv_id: str, submission_date: str):
    url = f"https://arxiv.org/abs/{arxiv_id}"
    try:
        res = session.get(url, timeout=10)
        if res.status_code != 200:
            return []
        soup = BeautifulSoup(res.text, "html.parser")
        history = soup.find("div", {"class": "submission-history"})
        if not history:
            return []
        dates = set()
        # Regex trích ngày “Wed, 13 Nov 2024 00:00:00 UTC”
        date_lines = re.findall(r"\[(v\d+)\]\s+(.*?)UTC", history.text)
        for _, text_date in date_lines:
            try:
                dt = datetime.strptime(text_date.strip(), "%a, %d %b %Y %H:%M:%S")
                d = dt.strftime("%Y-%m-%d")
            except ValueError:
              # fallback nếu format khác
                m = re.search(r"\d{1,2} \w+ \d{4}", text_date)
                if not m:
                    continue
                dt = datetime.strptime(m.group(), "%d %b %Y")
                d = dt.strftime("%Y-%m-%d")
            if d != submission_date:
                dates.add(d)
        return sorted(list(dates))
    except Exception:
        return []


"""Lấy metadata từ arxiv API (title, authors, date, category...)."""
def fetch_metadata(arxiv_id: str):
    try:
        search = arxiv.Search(id_list=[arxiv_id], max_results=1)
        result = next(arxiv.Client(page_size=1, delay_seconds=0.5).results(search))
        submission_date = result.published.strftime("%Y-%m-%d")
        revised_dates = fetch_revised_dates(arxiv_id, submission_date)
        cats = ", ".join(result.categories) if result.categories else "Unknown"
        metadata = {
            "title": result.title.strip(),
            "authors": [a.name for a in result.authors],
            "summary": result.summary.strip(),
            "submission_date": submission_date,
            "revised_dates": revised_dates,
            "publication_venue": result.journal_ref or "arXiv preprint",
            "pdf_url": result.pdf_url,
            "category": cats
        }
        return metadata, result
    except Exception as e:
        print(f"[ERROR] {arxiv_id}: {e}")
        return None, None


def get_gzip_original_name(gz_path: Path) -> str | None:
    """
    Đọc tên file gốc (FNAME) từ header .gz nếu có.
    Trả về None nếu không có tên hoặc lỗi định dạng.
    """
    try:
        with open(gz_path, "rb") as f:
            # Magic bytes
            if f.read(2) != b"\x1f\x8b":
                return None
            method = f.read(1)
            flg = ord(f.read(1))
            f.read(6)  # mtime + extra flags + OS

            # Nếu có FEXTRA (bit 2)
            if flg & 0x04:
                xlen = int.from_bytes(f.read(2), "little")
                f.read(xlen)

            # Nếu có FNAME (bit 3)
            if flg & 0x08:
                original_name = b""
                while True:
                    ch = f.read(1)
                    if not ch or ch == b"\x00":
                        break
                    original_name += ch
                return original_name.decode("utf-8", errors="ignore")

            # Nếu có FCOMMENT (bit 4)
            if flg & 0x10:
                while True:
                    if f.read(1) == b"\x00":
                        break

        return None
    except Exception:
        return None



def is_tar_gz(file_path: Path) -> bool:
    """
    Kiểm tra file .gz có phải là tar archive (tar.gz) hay không.
    → True nếu là .tar.gz
    → False nếu chỉ là .gz nén 1 file duy nhất (.tex, .bib, ...)
    """
    try:
        with gzip.open(file_path, 'rb') as f:
            header = f.read(512)  # đọc 512 byte đầu tiên
            return b'ustar' in header  # dấu hiệu đặc trưng của file tar
    except Exception:
        return False


def extract_file(tar_path: Path, dest_dir: Path):
    """
    Giải nén file nén (.tar.gz hoặc .gz) vào dest_dir, chỉ giữ .tex và .bib.
    Phân biệt loại file dựa trên phần mở rộng tạm thời (tar.gz hoặc gz).
    """
    print(f"    [INFO] Bắt đầu giải nén file: {tar_path.name}")
    if is_tar_gz(tar_path):
        # Xử lý .tar.gz (archive)
        try:
            with tarfile.open(tar_path, "r:gz") as tar:
                for member in tar.getmembers():    # duyệt từng entry trong tarball
                    name = member.name.lower()     # tên file viết thường
                    if any(name.endswith(ext) for ext in [".tex", ".bib"]):
                        tar.extract(member, path=dest_dir)     # chỉ extract .tex/.bib
        except Exception as e:
            print(f"    [WARN] Lỗi giải nén .tar.gz {tar_path.name}: {e}")

    else: # Nếu file có phần mở rộng .gz
        try:
            original_name = get_gzip_original_name(tar_path) # Đọc tên file gốc từ header GZIP
            if original_name is None:   # Nếu không có tên gốc trong header
                original_name = Path(tar_path.stem).name  # fallback dùng tên file nén bỏ đuôi .gz

            # Loại bỏ đường dẫn (nếu có)
            original_name = os.path.basename(original_name)
            output_path = dest_dir / original_name   # Tạo đường dẫn lưu file giải nén

            with gzip.open(tar_path, "rb") as f_in, open(output_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)        #  Giải nén nội dung từ .gz ra file thật
        except Exception as e:
            print(f"    [WARN] Lỗi giải nén .gz {tar_path.name}: {e}")


def measure_dir_size(path: Path) -> int:
    """Tính tổng dung lượng (byte) của thư mục."""
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += (Path(root) / f).stat().st_size
            except FileNotFoundError:
                pass
    return total



"""Tải tất cả phiên bản (v1, v2,...) và giải nén ra tex/."""
def download_versions(result, arxiv_id: str, save_root: Path):
    try:
        versions = fetch_versions_html(arxiv_id)
        print(f"    [INFO] Phát hiện {len(versions)} phiên bản cho {arxiv_id}")

        tex_root = save_root / "tex"
        safe_mkdir(tex_root)

        # biến đo kích thước source & trạng thái tải thành công hay không
        size_before = 0
        size_after = 0
        success_flag = 0

        for ver_label in versions:
            print(f"    [INFO] Đang tải source cho {arxiv_id}{ver_label} ...")
            source_url = f"https://arxiv.org/e-print/{arxiv_id}{ver_label}"
            version_folder = f"{arxiv_id.replace('.', '-')}{ver_label}"
            ver_path = tex_root / version_folder
            safe_mkdir(ver_path)

            try:
              # --- Tải file nén .tar.gz ---
                res = session.get(source_url, timeout=20)
                if res.status_code != 200:
                    print(f"    [WARN] {arxiv_id}{ver_label}: HTTP {res.status_code}, bỏ qua.")
                    continue

                head = res.content[:4]
                if not (head.startswith(b"\x1f\x8b") or head.startswith(b"PK")):
                    print(f"    [WARN] {arxiv_id}{ver_label}: Không phải file nén hợp lệ (startswith={head!r})")
                    continue

                tmp = save_root / f"{arxiv_id}{ver_label}.tar.gz"
                with open(tmp, "wb") as f:
                    f.write(res.content)

                 #  tính size trước khi lọc hình
                size_before += tmp.stat().st_size


                  # --- Giải nén ---
                tmp_extract = save_root / f"temp_extract_{ver_label}"
                safe_mkdir(tmp_extract)
                extract_file(tmp, tmp_extract)

                tex_count = bib_count = 0
                 # --- Lọc chỉ .tex và .bib ---
                for root, _, files in os.walk(tmp_extract):
                    for file in files:
                        src = Path(root) / file
                        ext = file.lower().strip()
                        if ext.endswith(".tex") or ext.endswith(".bib"):
                            shutil.move(str(src), ver_path / file)
                            if ext.endswith(".tex"):
                                tex_count += 1
                            else:
                                bib_count += 1
                os.remove(tmp)
                shutil.rmtree(tmp_extract, ignore_errors=True)

                # Nếu không có file thì xoá folder rỗng
                if tex_count == 0 and bib_count == 0:
                    shutil.rmtree(ver_path, ignore_errors=True)
                    print(f"    [INFO] Version {version_folder} không có file .tex/.bib → bỏ qua.")
                else:
                    success_flag = 1  # success nếu có tex/bib
                    print(f"    [DONE] {version_folder}: {tex_count} tex, {bib_count} bib")
            except Exception as e:
                print(f"    [ERROR] {arxiv_id}{ver_label}: {e}")
                shutil.rmtree(ver_path, ignore_errors=True)


        size_after = measure_dir_size(tex_root) # kích thước sau khi lọc hình
        
        # Xóa nếu thư mục tex trống
        if not any(tex_root.glob("**/*")):
            shutil.rmtree(tex_root, ignore_errors=True)
        
        # >>> BỔ SUNG: trả về dữ liệu thống kê
        return success_flag, size_before, size_after

    except Exception as e:
        print(f"[ERROR] Không thể xử lý versions của {arxiv_id}: {e}")
        return 0, 0, 0

# ---------------------- CHẠY SONG SONG ----------------------
def process_one(arxiv_id: str, delay: float = 2.0):
  """Xử lý 1 paper: lấy metadata + tải version."""
  print(f"\n[INFO] === {arxiv_id} ===")
    #  đo thời gian và RAM ban đầu
  t0 = time.time()
  process = psutil.Process(os.getpid())
  ram_before = process.memory_info().rss
  

  metadata, result = fetch_metadata(arxiv_id)
  if not metadata:
        print(f"[WARN] {arxiv_id}: bỏ qua.")
        return

  paper_dir = BASE_DIR / arxiv_id.replace(".", "-")
  safe_mkdir(paper_dir)

     # Ghi metadata.json
  with open(paper_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

     # Gọi tải các version của paper
  success_flag, size_before, size_after = download_versions(result, arxiv_id, paper_dir)
  time.sleep(delay)
  

    #  đo thời gian, RAM, disk
  runtime = time.time() - t0
  ram_after = process.memory_info().rss
  ram_used_mb = (ram_after - ram_before) / (1024 * 1024)
  disk_usage_mb = (size_before + size_after) / (1024 * 1024)

    # > ghi dòng thống kê vào CSV
  with open(STATS_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            arxiv_id, success_flag, size_before, size_after,
            round(runtime, 2), round(ram_used_mb, 2),
            round(disk_usage_mb, 2)
        ])

  print(f"[DONE] {arxiv_id} | success={success_flag} | runtime={runtime:.1f}s | RAM={ram_used_mb:.1f}MB | size_after={size_after/1e6:.2f}MB")



def main(start_id: int, end_id: int, delay: float = 2.0, parallelism: int = 4):
 """Chạy song song nhiều ID theo khoảng chỉ định."""
 prefix = START_MONTH[2:].replace("-", "")
 all_ids = [f"{prefix}.{i:05d}" for i in range(start_id, end_id + 1)]
 print(f"[INFO] Xử lý {len(all_ids)} paper ({parallelism} luồng song song)\n")

# ThreadPoolExecutor → chạy đa luồng
 with ThreadPoolExecutor(max_workers=parallelism) as executor:
        futures = [executor.submit(process_one, arxiv_id, delay) for arxiv_id in all_ids]
        for fut in as_completed(futures):
            fut.result()
 print("\n=== HOÀN TẤT ===")

# ---------------------- THỰC THI ----------------------
main(10222, 15211, delay=2.5, parallelism=8)


 [Giai đoạn 2] – Trích xuất References từ Semantic Scholar
-------------------------------------------------------------
Mục tiêu:

    • Dùng Semantic Scholar API để lấy danh sách bài báo được trích dẫn (references) cho từng paper đã tải ở Giai đoạn 1.

    • Lưu kết quả dưới dạng references.json (key = arXiv ID, value = metadata của reference).

    • Tự động thêm API key, retry, và delay ngẫu nhiên để tránh lỗi 429 (rate-limit).

    • Chạy song song trên nhiều paper để tăng tốc độ xử lý.

Đầu ra:
    📂 <BASE_DIR>/<arxiv_id>/

        └── references.json

Thông tin kỹ thuật:
    - API: Semantic Scholar Graph API v1

    - Trường dữ liệu: title, authors, publicationDate, venue, semantic_scholar_id, corpusId, fieldsOfStudy

    - Ngôn ngữ: Python 3.12

    - Thư viện: requests, concurrent.futures



In [ ]:

from google.colab import drive

drive.mount('/content/mydrive')

import os, json, time, requests, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========================  CẤU HÌNH ========================
# Thư mục chứa dữ liệu đầu vào (đã crawl từ Giai đoạn 1)
BASE_DIR = Path("/content/mydrive/MyDrive/23120257")
BASE_DIR.mkdir(parents=True, exist_ok=True)



# ========================  SEMANTIC SCHOLAR API ========================

# Nhập API key của bạn tại đây:
API_KEY = "AQhxAYRK4e6rL5J7gNEwa7xDqGxYbFFy6AI0d3np"  #  thay bằng key thật

# Tạo session HTTP chung
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "x-api-key": API_KEY  # <--- thêm dòng này
})

STATS_FILE_REF = BASE_DIR / "statistics_program2.csv"

# Endpoint chính của Semantic Scholar API
SEMANTIC_API = "https://api.semanticscholar.org/graph/v1/paper/arXiv:{}"

# Tạo file CSV nếu chưa có
if not STATS_FILE_REF.exists():
    with open(STATS_FILE_REF, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "arxiv_id", "success_rate", "ref_numbers",
            "runtime_sec", "ram_usage_mb", "disk_usage_mb"
        ])

# ======================== HÀM CHÍNH GỌI API ========================

def fetch_references(arxiv_id: str, max_retries: int = 5):
    """
    Gọi Semantic Scholar API để lấy danh sách references cho 1 paper cụ thể.
    - Nếu HTTP 429 → tự động chờ ngẫu nhiên rồi thử lại.
    - Trả về dict chứa các paper tham chiếu có arXiv ID.
    """

    url = SEMANTIC_API.format(arxiv_id)
    params = {
        "fields": (
            "references.title,references.authors,references.externalIds,"
            "references.venue,references.paperId,references.publicationDate,"
            "references.corpusId,references.fieldsOfStudy"
        )
    }

    for attempt in range(1, max_retries + 1):
        try:
            res = session.get(url, params=params, timeout=20)

            #  Thành công → phân tích JSON trả về
            if res.status_code == 200:
                data = res.json() or {}
                refs_raw = data.get("references", [])
                refs = {}
            # Lọc ra những reference có arXiv ID
                for r in refs_raw:
                    ext = r.get("externalIds") or {}
                    ref_arx = ext.get("ArXiv") or ext.get("arXiv")
                    if not ref_arx:
                        continue

                    key = ref_arx.replace(".", "-")
                    refs[key] = {
                        "title": r.get("title", "").strip(),
                        "authors": [a.get("name") for a in (r.get("authors") or []) if isinstance(a, dict)],
                        "publication_date": r.get("publicationDate") or "",
                        "venue": r.get("venue") or "arXiv reference",
                        "semantic_scholar_id": r.get("paperId") or "",
                        "corpus_id": r.get("corpusId") or "",
                        "category": ", ".join(r.get("fieldsOfStudy", []) or [])
                    }

                return refs

            #  Nếu bị rate limit
            elif res.status_code == 429:
                wait_time = random.uniform(6, 10)
                print(f"[WARN] {arxiv_id}: HTTP 429 (Rate limit) → chờ {wait_time:.1f}s rồi thử lại ({attempt}/{max_retries})")
                time.sleep(wait_time)
                continue
           #  Lỗi HTTP khác (404, 500, …)
            else:
                print(f"[WARN] {arxiv_id}: HTTP {res.status_code}")
                return {}
          # Nếu gặp lỗi mạng hoặc timeout
        except Exception as e:
            print(f"[ERROR] {arxiv_id} (thử {attempt}/{max_retries}): {e}")
            time.sleep(5)

    # Nếu hết số lần retry mà vẫn lỗi
    print(f"[FAIL] {arxiv_id}: quá số lần retry ({max_retries})")
    return {}


# ======================== HÀM XỬ LÝ MỖI PAPER ========================

def process_one_reference(folder: Path, delay: float = 3.0):
  """
    Tạo file `references.json` cho 1 paper cụ thể.
    - Lấy ID từ tên thư mục 
    - Bỏ qua nếu file đã tồn tại.
     Các chỉ số thống kê:
      - success_rate: 1 nếu có reference, 0 nếu không
      - ref_numbers: tổng số reference lấy được
      - runtime_sec: thời gian xử lý 1 paper
      - ram_usage_mb: dung lượng RAM sử dụng
      - disk_usage_mb: kích thước file references.json
    """
   # Nếu đã có file thì bỏ qua
  arxiv_id = folder.name.replace("-", ".")
  ref_file = folder / "references.json"
    
    # Ghi nhận thời gian và RAM ban đầu
  t0 = time.time()
  process = psutil.Process(os.getpid())
  ram_before = process.memory_info().rss
  success_flag = 0
  ref_numbers = 0 
  disk_usage = 0
  

  if ref_file.exists():
        print(f"[SKIP] {arxiv_id}: đã có references.json")
        return
  
 # Gọi API để lấy dữ liệu
  refs = fetch_references(arxiv_id)

# Ghi ra file JSON
  with open(ref_file, "w", encoding="utf-8") as f:
        json.dump(refs, f, indent=2, ensure_ascii=False)

# Đếm số reference và đánh dấu thành công
  ref_numbers = len(refs)
  if ref_numbers > 0:
        success_flag = 1

# Tính toán RAM, disk, runtime
  runtime = time.time() - t0
  ram_after = process.memory_info().rss
  ram_used_mb = (ram_after - ram_before) / (1024 * 1024)
  disk_usage = ref_file.stat().st_size if ref_file.exists() else 0
  disk_usage_mb = disk_usage / (1024 * 1024)

   #  BỔ SUNG: Ghi dòng thống kê vào CSV
  with open(STATS_FILE_REF, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            arxiv_id, success_flag, ref_numbers,
            round(runtime, 2), round(ram_used_mb, 2),
            round(disk_usage_mb, 2)
        ])

    # >>> Ghi log ra màn hình
  print(f"[DONE] {arxiv_id} | success={success_flag} | refs={ref_numbers} | runtime={runtime:.2f}s | RAM={ram_used_mb:.2f}MB | disk={disk_usage_mb:.2f}MB")

  print(f"[DONE] {arxiv_id}: {len(refs)} references")
  time.sleep(delay)

# ======================== HÀM XỬ LÝ SONG SONG ========================
def run_program2(parallelism=8, delay=5.0):
    """
    📘 Hàm: run_program2
    --------------------
    Chức năng:
      - Duyệt qua tất cả các thư mục paper trong BASE_DIR
      - Gọi process_one_references() cho mỗi paper song song bằng ThreadPoolExecutor
      - Ghi log và tổng kết sau khi hoàn tất
    """
    folders = [p for p in BASE_DIR.iterdir() if p.is_dir()]
    print(f"[INFO] Tổng số paper cần xử lý: {len(folders)}")

    #  Chạy đa luồng
    with ThreadPoolExecutor(max_workers=parallelism) as ex:
        futs = [ex.submit(process_one_reference, f) for f in folders]
        for f in as_completed(futs):
            f.result()
            time.sleep(delay)

    print("\n=== PROGRAM 2 HOÀN TẤT ===")

# ======================== THỰC THI CHÍNH ========================
run_program2()


# ========================  CELL 3 – PHÂN TÍCH & VẼ BIỂU ĐỒ ========================
"""
 CHỨC NĂNG CHÍNH:
-------------------
Cell này tổng hợp và trực quan hóa kết quả thu được từ hai chương trình trước:

 CHƯƠNG TRÌNH 1 – Arxiv Crawler:
   - Đọc dữ liệu thống kê từ file statistics_program1.csv
   - Tính toán các chỉ số:
       • Tổng số bài báo xử lý
       • Số bài crawl thành công (success_rate = 1)
       • Tỷ lệ thành công (overall success rate)
       • Kích thước file trung bình trước và sau khi loại bỏ hình ảnh (figures)
       • Thời gian trung bình và tổng thời gian crawl
       • Maximum RAM, maximum disk
       • Trung bình RAM sử dụng cho 1 bài

 CHƯƠNG TRÌNH 2 – References Crawler:
   - Đọc dữ liệu thống kê từ file statistics_program2.csv
   - Tính toán các chỉ số:
       • Số bài reference trung bình mỗi paper
       • Tỷ lệ thành công (success_rate = 1)
       • Thời gian trung bình và tổng thời gian crawl references
       • Maximum RAM, maximum disk
       • Trung bình RAM sử dụng cho 1 bài

 TỔNG HỢP CẢ 2 CHƯƠNG TRÌNH:
   - Tổng thời gian crawl (1 + 2)
   - Thời gian trung bình xử lý 1 bài (1 + 2)
   - Tổng RAM và RAM trung bình sử dụng (1 + 2)
   - Ghi toàn bộ kết quả vào file summary_statistics.txt trong BASE_DIR

 VẼ BIỂU ĐỒ:
   - Biểu đồ 1 (Pie chart): Tỷ lệ crawl thành công (success_rate = 1/0) của chương trình 1
   - Biểu đồ 2 (Pie chart): Tỷ lệ crawl reference thành công (success_rate = 1/0) của chương trình 2
   - Biểu đồ 3 (Line chart): RAM sử dụng theo thời gian của cả hai chương trình
   
 KẾT QUẢ:
-----------
• File text summary_statistics.txt ghi toàn bộ thông tin tổng hợp.
• Hiển thị 3 biểu đồ minh họa cho tỷ lệ thành công và mức tiêu thụ RAM.
• Toàn bộ dữ liệu được lấy trực tiếp từ hai file CSV của Cell 1 và Cell 2.

"""


In [ ]:
# ========================  CELL 3 – PHÂN TÍCH & VẼ BIỂU ĐỒ ========================
# Chức năng:
#   - Đọc dữ liệu thống kê từ statistics_program1.csv và statistics_program2.csv
#   - Tính toán các chỉ số tổng hợp cho chương trình 1 & 2
#   - Ghi toàn bộ thông tin ra file text summary_statistics.txt
#   - Vẽ 3 biểu đồ: 2 pie chart (success rate), 1 line chart (RAM theo thời gian)

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("/content/mydrive/MyDrive/23120257")
CSV1 = BASE_DIR / "statistics_program1.csv"
CSV2 = BASE_DIR / "statistics_program2.csv"
SUMMARY_FILE = BASE_DIR / "summary_statistics.txt"

def get_dir_size(path: Path) -> float:
    """Tính tổng dung lượng thư mục (MB), bao gồm tất cả file con."""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = Path(dirpath) / f
            if fp.is_file():
                total += fp.stat().st_size
    return total / (1024 * 1024)  # Đổi sang MB

def get_final_output_size(base_dir: Path) -> float:
    """Tính dung lượng output cuối cùng (MB), trừ 2 file CSV thống kê."""
    total_size = get_dir_size(base_dir)
    csv_files = [
        BASE_DIR / "statistics_program1.csv",
        BASE_DIR / "statistics_program2.csv",
        BASE_DIR / "summary_statistics.txt"
    ]
    csv_total = sum(f.stat().st_size for f in csv_files if f.exists()) / (1024 * 1024)
    return total_size - csv_total

final_output_disk = get_final_output_size(BASE_DIR)

# ======================== ĐỌC DỮ LIỆU ========================
df1 = pd.read_csv(CSV1)
df2 = pd.read_csv(CSV2)

# ======================== PHÂN TÍCH CHƯƠNG TRÌNH 1 ========================
total_papers_1 = len(df1)
success_papers_1 = df1["success_rate"].sum()
overall_success_rate_1 = success_papers_1 / total_papers_1 if total_papers_1 else 0

avg_size_before = df1["size_before_figures"].mean()
avg_size_after = df1["size_after_figures"].mean()
avg_runtime_1 = df1["runtime_sec"].mean()
total_runtime_1 = df1["runtime_sec"].sum()
max_ram_1 = df1["ram_usage_mb"].max()
max_disk_1 = df1["disk_usage_mb"].max()
avg_ram_1 = df1["ram_usage_mb"].mean()

# ======================== PHÂN TÍCH CHƯƠNG TRÌNH 2 ========================
total_papers_2 = len(df2)
success_papers_2 = df2["success_rate"].sum()
overall_success_rate_2 = success_papers_2 / total_papers_2 if total_papers_2 else 0

avg_ref_per_paper = df2["ref_numbers"].mean()
avg_runtime_2 = df2["runtime_sec"].mean()
total_runtime_2 = df2["runtime_sec"].sum()
max_ram_2 = df2["ram_usage_mb"].max()
max_disk_2 = df2["disk_usage_mb"].max()
avg_ram_2 = df2["ram_usage_mb"].mean()

# ======================== TỔNG HỢP CẢ 2 CHƯƠNG TRÌNH ========================
total_runtime_all = total_runtime_1 + total_runtime_2
avg_runtime_all = avg_runtime_1 + avg_runtime_2
max_ram_all = max_ram_1 + max_ram_2
avg_ram_all = avg_ram_1 + avg_ram_2

# ======================== GHI FILE SUMMARY ========================
with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    f.write("===== PROGRAM 1 – ARXIV CRAWLER =====\n")
    f.write(f"• Tổng số bài báo xử lý: {total_papers_1}\n")
    f.write(f"• Số bài crawl thành công: {success_papers_1}\n")
    f.write(f"• Overall success rate: {overall_success_rate_1:.2%}\n")
    f.write(f"• Kích thước trung bình trước khi loại figures: {avg_size_before:.2f} bytes\n")
    f.write(f"• Kích thước trung bình sau khi loại figures: {avg_size_after:.2f} bytes\n")
    f.write(f"• Thời gian trung bình xử lý 1 bài: {avg_runtime_1:.2f} s\n")
    f.write(f"• Tổng thời gian crawl: {total_runtime_1:.2f} s\n")
    f.write(f"• Maximum RAM sử dụng: {max_ram_1:.2f} MB\n")
    f.write(f"• Maximum Disk sử dụng: {max_disk_1:.2f} MB\n")
    f.write(f"• Trung bình RAM cho 1 bài: {avg_ram_1:.2f} MB\n\n")

    f.write("===== PROGRAM 2 – REFERENCES CRAWLER =====\n")
    f.write(f"• Tổng số bài xử lý: {total_papers_2}\n")
    f.write(f"• Số bài references thành công: {success_papers_2}\n")
    f.write(f"• Overall success rate: {overall_success_rate_2:.2%}\n")
    f.write(f"• Số references trung bình mỗi paper: {avg_ref_per_paper:.2f}\n")
    f.write(f"• Thời gian trung bình xử lý 1 bài: {avg_runtime_2:.2f} s\n")
    f.write(f"• Tổng thời gian crawl references: {total_runtime_2:.2f} s\n")
    f.write(f"• Maximum RAM sử dụng: {max_ram_2:.2f} MB\n")
    f.write(f"• Maximum Disk sử dụng: {max_disk_2:.2f} MB\n")
    f.write(f"• Trung bình RAM cho 1 bài: {avg_ram_2:.2f} MB\n\n")

    f.write("===== TỔNG HỢP CẢ 2 CHƯƠNG TRÌNH =====\n")
    f.write(f"• Tổng thời gian crawl (1+2): {total_runtime_all:.2f} s\n")
    f.write(f"• Thời gian trung bình 1 bài (1+2): {avg_runtime_all:.2f} s\n")
    f.write(f"• Tổng RAM sử dụng (1+2): {max_ram_all:.2f} MB\n")
    f.write(f"• Trung bình RAM mỗi bài (1+2): {avg_ram_all:.2f} MB\n")
    f.write(f"• Final output storage size: {final_output_disk:.2f} MB\n")
print(f" Đã ghi báo cáo tổng hợp vào {SUMMARY_FILE}")

# ======================== VẼ BIỂU ĐỒ ========================

# --- Pie chart – Success rate chương trình 1 ---
plt.figure(figsize=(5,5))
plt.pie(
    [success_papers_1, total_papers_1 - success_papers_1],
    labels=["Success (1)", "Fail (0)"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#F44336"]
)
plt.title("Tỉ lệ crawl thành công – Program 1 (Arxiv)")
plt.show()

# --- Pie chart – Success rate chương trình 2 ---
plt.figure(figsize=(5,5))
plt.pie(
    [success_papers_2, total_papers_2 - success_papers_2],
    labels=["Success (1)", "Fail (0)"],
    autopct="%1.1f%%",
    colors=["#2196F3", "#FFC107"]
)
plt.title("Tỉ lệ crawl references thành công – Program 2 (Semantic Scholar)")
plt.show()

# --- Line chart – RAM sử dụng theo thời gian (Program 1 + 2) ---
plt.figure(figsize=(8,5))
plt.plot(df1["ram_usage_mb"], label="Program 1 – Arxiv", marker="o")
plt.plot(df2["ram_usage_mb"], label="Program 2 – References", marker="x")
plt.xlabel("Paper index")
plt.ylabel("RAM sử dụng (MB)")
plt.title("Biểu đồ RAM sử dụng theo thời gian (Program 1 & 2)")
plt.legend()
plt.grid(True)
plt.show()
